In [1]:
import pandas as pd
import numpy as np

# --- 1. PARÂMETROS DO SISTEMA ---
Vbar = 100       # Vdc
f_sw = 10000     # 12 kHz
Ts = 1/f_sw        # Período de amostragem/comutação (Equação 5)

print(f"--- PARÂMETROS DO SISTEMA ---")
print(f"Vdc:  {Vbar}V")
print(f"f_sw: {f_sw}Hz")
print(f"Ts:   {Ts*1e6:.2f} us\n")

# --- 2. Vetores reais: [S1, S3, VA, VB, VAB]
vetor_0 = [0, 0, 0.0,  0.0,   0.0]
vetor_1 = [1, 0, Vbar, 0.0,   Vbar]
vetor_2 = [0, 1, 0.0,  Vbar, -Vbar]
vetor_3 = [1, 1, Vbar, Vbar,  0.0]

print(f"--- VETORES REAIS [S1, S3, VA, VB, VAB] ---")
print(f"  vetor_0: {vetor_0}")
print(f"  vetor_1: {vetor_1}")
print(f"  vetor_2: {vetor_2}")
print(f"  vetor_3: {vetor_3}\n")

# 3 -- Vetores normalizados (v'_x = vx/Vdc)
vetor_0_n = [vetor_0[0], vetor_0[1], vetor_0[2]/Vbar, vetor_0[3]/Vbar, vetor_0[4]/Vbar]
vetor_1_n = [vetor_1[0], vetor_1[1], vetor_1[2]/Vbar, vetor_1[3]/Vbar, vetor_1[4]/Vbar]
vetor_2_n = [vetor_2[0], vetor_2[1], vetor_2[2]/Vbar, vetor_2[3]/Vbar, vetor_2[4]/Vbar]
vetor_3_n = [vetor_3[0], vetor_3[1], vetor_3[2]/Vbar, vetor_3[3]/Vbar, vetor_3[4]/Vbar]

print(f"--- VETORES NORMALIZADOS [S1, S3, VA', VB', VAB'] ---")
print(f"  vetor_0_n: {vetor_0_n}")
print(f"  vetor_1_n: {vetor_1_n}")
print(f"  vetor_2_n: {vetor_2_n}")
print(f"  vetor_3_n: {vetor_3_n}\n")

# --- 4. ENTRADA DE TENSÃO (Vetor de Comando)
u_cmd = 60                           # tensão a ser produzida pelo inversor
u_cmd_L = u_cmd / Vbar                  # Tensão normalizada

print(f"--- PASSO 1: DEFINIÇÃO DO VETOR DE COMANDO ---")
print(f"U_cmd desejado: {u_cmd}V")
print(f"u_cmd_L (Normalizado): {u_cmd_L}\n")

# --- 5. IDENTIFICAÇÃO DO SETOR ---
if u_cmd_L >= 0:
    setor = 1
    vec_ativo_n = vetor_1_n
    v_label = "v1"
    if u_cmd_L >= 1:
      print("ERROR: U_cmd_L maior que 1")
      u_cmd_L =1
      print(f"Novo u_cmd_L (Normalizado): {u_cmd_L}\n")
else:
    setor = 2
    vec_ativo_n = vetor_2_n
    v_label = "v2"
    if u_cmd_L <= -1:
      print("ERROR: U_cmd_L menor que 1")
      u_cmd_L =-1
      print(f"Novo u_cmd_L (Normalizado): {u_cmd_L}\n")

print(f"--- PASSO 2: PLANOS DE SEPARAÇÃO E LIMITES ---")
print(f"Setor: {setor} (Vetor ativo selecionado: {v_label})")
print(f"Vetor {v_label} normalizado: {vec_ativo_n}\n")

# --- 6. CÁLCULO DOS TEMPOS -

v1= vec_ativo_n[4]
M1 = 1 / v1
delta_t1 = Ts * M1 * u_cmd_L
t_nulo_total = Ts - delta_t1

print(f"--- PASSO 3: MATRIZ DE DECOMPOSIÇÃO E TEMPOS ---")
print(f"v1: {v1}")
print(f"M{setor} = {M1}")
print(f"Duração do Vetor Ativo (delta_t1): {delta_t1*1e6:.3f} us")
print(f"Duração dos Vetores Nulos (t_nulo): {t_nulo_total*1e6:.3f} us\n")

# --- 5. SEQUÊNCIA DE COMUTAÇÃO SIMÉTRICA (Tabela 3) ---
t_v_at_metade = delta_t1 / 2
t_v3 = t_nulo_total / 2
t_v0 = t_v3 / 2 #Pois aparece dois TV0 logo 2xtv0 = tv3

# Definindo a sequência baseada no setor
if setor == 1:
    seq_labels = ["v0", "v1", "v3", "v1", "v0"]
    seq_vecs = [vetor_0_n, vetor_1_n, vetor_3_n, vetor_1_n, vetor_0_n]
else:
    seq_labels = ["v0", "v2", "v3", "v2", "v0"]
    seq_vecs = [vetor_0_n, vetor_2_n, vetor_3_n, vetor_2_n, vetor_0_n]

duracoes = [t_v0, t_v_at_metade, t_v3, t_v_at_metade, t_v0]

sequencia = []
for label, vec, dur in zip(seq_labels, seq_vecs, duracoes):
    sequencia.append({
        "Passo": label,
        "S1": vec[0],
        "S3": vec[1],
        "VAB'": vec[4],
        "Duração (us)": dur*1e6
    })

df_seq = pd.DataFrame(sequencia)
print(f"--- PASSO 4: SEQUÊNCIA DE COMUTAÇÃO (Tabela 3) ---")
print(df_seq.to_string(index=False))


ModuleNotFoundError: No module named 'pandas'